<a id='1-2'></a>
### 
This function is designed to handle multiple input messages in a conversational context. The input format is a dictionary with two keys:

1. 'role' - the role that the message is being passed (usually assistant, system or user)
2. 'content' - the prompt

#### Parameters:

- `messages` (List[Dict]): A list of dictionaries, each containing 'role' and 'content' keys to represent each message in the conversation.
- `max_tokens` (int): Determines token limit for the response.
- `model` (str): Model to be used, default is `"meta-llama/Llama-3.2-3B-Instruct-Turbo"`.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
open_router_api_key = os.getenv("OPENROUTER_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")

# Enable LangChain's tracing to monitor and debug chain executions
os.environ["LANGCHAIN_TRACING_V2"] = "true"

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

if not open_router_api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

if not langsmith_api_key:
    raise ValueError("LANGSMITH_API_KEY not found in .env file")

print("✓ Environment loaded successfully")
print(f"✓ OpenAI API Key: {openai_api_key[:20]}..." if openai_api_key else "✗ OpenAI API Key not found")
print(f"✓ OpenRouter API Key: {open_router_api_key[:20]}..." if open_router_api_key else "✗ OpenRouter API Key not found")
print(f"✓ LangSmith API Key: {langsmith_api_key[:20]}..." if langsmith_api_key else "✗ LangSmith API Key not found")

✓ Environment loaded successfully
✓ OpenAI API Key: sk-proj-Xo-lT2Oxn0cM...
✓ OpenRouter API Key: sk-or-v1-fa41df1a84e...
✓ LangSmith API Key: lsv2_pt_8d4edf0310f0...


In [2]:
from IPython.display import Markdown, display

def print_markdown(text):
    """Display formatted markdown text in Jupyter notebooks"""
    display(Markdown(text))

# Show passed message
print_markdown("""
✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!
""")


✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!


In [4]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [5]:
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



In [7]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


In [12]:
from openai import OpenAI

# Initialize the client (uses OPENAI_API_KEY environment variable by default)
client = OpenAI()

def generate_response(prompt: str, role: str):
        # Make a chat completion request
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "your helpful assistant. based given prompt, answer the user query accurately."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=150
    )

    # Print the response
    print(response.choices[0].message.content)

In [13]:
#print_markdown("### Generating response for the query: 'What is the most expensive house?'")
#print_markdown(generate_response(generate_prompt("What is the most expensive house?", houses = house_data), role="user"))
generate_response(generate_prompt("What is the most expensive house?", houses = house_data), role="user")

The most expensive house is located at 456 Elm Avenue, Shelbyville, TN 37160, priced at $320,000.
